# ByteEmbed — feasibility (Colab A100)

Tokenizer-free byte-level retriever distilled from a frozen multilingual subword teacher.

**Before running:** Runtime → Change runtime type → **A100 GPU**. Then Runtime → **Run all**.
Self-contained: installs deps, downloads data, distills, evaluates robustness.

In [ ]:
!nvidia-smi

In [ ]:
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
!git clone -q $REPO
%cd embedding-research
!pip install -q -r requirements-cloud.txt unidecode
!pip install -q -e .

In [ ]:
!python scripts/check_env.py

In [ ]:
from byte_embed.run_feasibility import run

# A100 budget. Raise n_per_lang / steps for a stronger signal; lower for a quick look.
results = run(steps=2000, batch=64,
              out='results/byte_embed_colab.json',
              save_student='results/byte_student.pt')

## Reading the result

- **align_cos** high → the byte student reproduces the teacher's embeddings.
- **xP@1(stu) vs xP@1(tea)** → cross-lingual retrieval; the student should approach the teacher.
- **robustness**: student stability **> teacher** under `romanize` / `spelling` → the byte-level robustness hypothesis (H2) holds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

h = results['train_loss']
plt.figure(figsize=(5,3))
plt.plot([x['step'] for x in h], [x['loss'] for x in h])
plt.xlabel('step'); plt.ylabel('cosine distillation loss')
plt.title('ByteEmbed distillation'); plt.tight_layout(); plt.show()

rob = results['robustness']
perts = ['diacritics','romanize','spelling']
stu = [np.mean([rob[l][p]['student'] for l in rob]) for p in perts]
tea = [np.mean([rob[l][p]['teacher'] for l in rob]) for p in perts]
x = np.arange(len(perts)); w = 0.35
plt.figure(figsize=(5,3))
plt.bar(x-w/2, stu, w, label='byte student')
plt.bar(x+w/2, tea, w, label='subword teacher')
plt.xticks(x, perts); plt.ylabel('cosine stability')
plt.legend(); plt.title('Orthographic robustness'); plt.tight_layout(); plt.show()

## Notes
- Laptop proof-of-life (12 GB): `python -m byte_embed.run_feasibility --smoke`.
- This is the cheap feasibility check, **not** the full iso-compute / fertility study — see `byte_embed/README.md`.